# Gov Actuator

**Input** (notebook parameters, JSON string `request_json`):

```json
{ "correlation_id": "...", "request_id": "...", "actor": "alkorn@...",
  "dry_run": true,
  "binding": { "kind": "entra_group_member", "module": "entra",
               "target_id": "ws-pilot", "target_type": "Workspace",
               "principal_id": "...", "role": "Contributor",
               "writable": true } }
```

**Output** (`exitValue`): the actuator contract —
`{ ok, dry_run, before, after, verified, verify_after_s, detail, error }`.

A refused call returns `ok:false` with `error:"gate:<name>"` **and still writes
`gov_audit`**. A refusal nobody recorded is indistinguishable from a write that
never happened.

In [ ]:
lakehouse_name = "governance_lh"
request_json = "{}"
# Escape hatch for a customer running the notebook by hand: it changes nothing
# about the gates, only whether the audit row is persisted.
persist_audit = True
# Power Platform writes authenticate as the registered management app. A human
# Power Platform Administrator must have run `New-PowerAppManagementApp` once —
# a service principal cannot register itself.
key_vault_uri = ""
sp_client_id_secret = "gov-pp-client-id"
sp_secret_secret = "gov-pp-secret"
tenant_id = ""

In [ ]:
# --- inlined from collectors/shape_common.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Sequence


def utcnow() -> datetime:
    return datetime.now(timezone.utc)


def new_run_id() -> str:
    return str(uuid.uuid4())


def as_str(value: Any) -> str | None:
    """Normalise an API scalar to a string, preserving a real absence as None.

    Collector tables are all-string on purpose: every plane has its own id
    format, and coercing them into typed columns is how a join silently starts
    returning nothing.
    """
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def as_json(value: Any) -> str | None:
    """Stable JSON for a blob column. Sorted keys so diffs are meaningful."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def stamp(rows: Iterable[dict], run_id: str, scanned_at: datetime | None = None) -> list[dict]:
    """Attach the run provenance every `gov_actual_*` row carries."""
    when = scanned_at or utcnow()
    out = []
    for row in rows:
        enriched = dict(row)
        enriched["run_id"] = run_id
        enriched["scanned_at"] = when
        out.append(enriched)
    return out


class RunLedger:
    """Accumulates what a collector did, for the `gov_runs` row and the app.

    Errors are first-class: a collector that quietly drops an unreadable object
    produces a governance report that is wrong in the most dangerous direction —
    it under-reports access.
    """

    def __init__(self, collector: str, module: str, tier: str) -> None:
        self.run_id = new_run_id()
        self.collector = collector
        self.module = module
        self.tier = tier
        self.started_at = utcnow()
        self.finished_at: datetime | None = None
        self.errors: list[dict[str, str]] = []
        self.counts: dict[str, int] = {}

    def count(self, table: str, n: int) -> None:
        self.counts[table] = self.counts.get(table, 0) + n

    def error(self, scope: str, exc: BaseException | str) -> None:
        self.errors.append(
            {
                "scope": scope,
                "type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
                "message": str(exc),
            }
        )

    def finish(self) -> dict:
        self.finished_at = utcnow()
        return {
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "n_objects": sum(self.counts.values()),
            "n_errors": len(self.errors),
            "error_json": as_json(self.errors) if self.errors else None,
            "duration_s": (self.finished_at - self.started_at).total_seconds(),
        }

    def exit_value(self, *, dry_run: bool) -> dict:
        """Actuator-contract-shaped result (PLAN.md §14) for the app to parse."""
        return {
            "ok": True,
            "dry_run": dry_run,
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "counts": dict(self.counts),
            "n_errors": len(self.errors),
            "errors": self.errors[:20],
            "finished_at": (self.finished_at or utcnow()).isoformat(),
        }


def safe_each(
    items: Sequence[Any],
    fn: Callable[[Any], list[dict]],
    ledger: RunLedger,
    scope_of: Callable[[Any], str],
) -> list[dict]:
    """Map `fn` over `items`, recording per-item failures instead of raising."""
    rows: list[dict] = []
    for item in items:
        try:
            rows.extend(fn(item))
        except Exception as exc:  # noqa: BLE001 — a collector must never hard-fail
            ledger.error(scope_of(item), exc)
    return rows

In [ ]:
# --- inlined from collectors/runtime.py (unit-tested offline) ---
from __future__ import annotations

import json
import time
from typing import Any, Callable


class RestError(RuntimeError):
    def __init__(self, status: int, url: str, body: str) -> None:
        super().__init__(f"{status} {url}: {body[:400]}")
        self.status = status
        self.url = url


def fabric_client():
    """A `sempy` REST client for Fabric / Power BI, under the running identity."""
    import sempy.fabric as fabric  # type: ignore

    return fabric.FabricRestClient()


def rest_get(client, path: str, *, retries: int = 4) -> dict[str, Any]:
    """GET with backoff on 429/5xx.

    Admin APIs are rate-limited (25 req/min on some tenant-setting endpoints), and
    a nightly crawl that gives up on the first 429 silently under-reports — which
    is the worst possible failure mode for a governance inventory.
    """
    delay = 2.0
    last: Exception | None = None
    for _ in range(retries):
        response = client.get(path)
        if response.status_code == 200:
            return response.json() if response.text else {}
        if response.status_code in (429, 500, 502, 503, 504):
            retry_after = response.headers.get("Retry-After")
            time.sleep(float(retry_after) if retry_after else delay)
            delay = min(delay * 2, 60)
            last = RestError(response.status_code, path, response.text)
            continue
        raise RestError(response.status_code, path, response.text)
    raise last or RestError(0, path, "exhausted retries")


def graph_token(scope: str = "https://graph.microsoft.com/.default") -> str:
    """Delegated Graph token for the identity the notebook runs as."""
    import notebookutils  # type: ignore

    return notebookutils.credentials.getToken(scope)


def graph_get(token: str, url: str, *, retries: int = 4) -> dict[str, Any]:
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    delay = 2.0
    for _ in range(retries):
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        try:
            with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 500, 502, 503, 504):
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc
    raise RestError(0, url, "exhausted retries")


def graph_call(token: str, method: str, url: str, body: dict | None = None) -> dict[str, Any]:
    """Graph request with a method — the write-capable sibling of `graph_get`.

    Deliberately **not** retried on 5xx: a POST that may have partially applied
    must not be replayed blindly. The actuator's read-before-write makes a
    retry safe only after re-reading, and that is the caller's decision.
    """
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header("Authorization", f"Bearer {token}")
    if data is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as exc:
        raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc


def fabric_call(client, method: str, path: str, body: dict | None = None) -> dict[str, Any]:
    """Fabric REST with a method, through the `sempy` client.

    Same no-retry stance as `graph_call`, for the same reason.
    """
    verb = method.upper()
    if verb == "GET":
        response = client.get(path)
    elif verb == "POST":
        response = client.post(path, json=body or {})
    elif verb == "PATCH":
        response = client.patch(path, json=body or {})
    elif verb == "DELETE":
        response = client.delete(path)
    else:
        raise ValueError(f"unsupported method {method}")

    if response.status_code not in (200, 201, 202, 204):
        raise RestError(response.status_code, path, response.text)
    return response.json() if response.text else {}


def write_table(
    spark,
    lakehouse: str,
    table: str,
    rows: list[dict],
    *,
    dry_run: bool,
    log: Callable[[str, str, str], None],
) -> int:
    """Overwrite one `gov_actual_*` table with this run's rows.

    Overwrite, not append: these tables are a *snapshot of current reality*, and
    the run ledger plus `gov_audit` carry the history. An append-only actual-state
    table is how a drift engine starts comparing against last month.
    """
    if dry_run:
        log(table, "Planned", f"{len(rows)} rows")
        return len(rows)
    if not rows:
        log(table, "Skipped (no permission)", "no rows collected")
        return 0
    try:
        df = spark.createDataFrame(rows)
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            f"{lakehouse}.{table}"
        )
        log(table, "Created", f"{len(rows)} rows")
        return len(rows)
    except Exception as exc:  # noqa: BLE001
        log(table, "Failed", f"{type(exc).__name__}: {exc}")
        return 0


def write_run_row(spark, lakehouse: str, summary: dict, *, dry_run: bool) -> None:
    if dry_run:
        return
    try:
        spark.createDataFrame([summary]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse}.gov_runs")
    except Exception as exc:  # noqa: BLE001
        print(f"gov_runs append failed: {exc}")


def finish(ledger, spark, lakehouse: str, *, dry_run: bool) -> str:
    summary = ledger.finish()
    write_run_row(spark, lakehouse, summary, dry_run=dry_run)
    result = ledger.exit_value(dry_run=dry_run)
    try:
        import notebookutils  # type: ignore

        notebookutils.notebook.exit(json.dumps(result))
    except ImportError:
        print(json.dumps(result, indent=2))
    return json.dumps(result)

In [ ]:
# --- inlined from collectors/gates.py (unit-tested offline) ---
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from typing import Any, Iterable

DRY_RUN_VALIDITY_DAYS = 30

#: Roles this tool must never grant, in any plane, under any configuration.
#: Compared case-insensitively after trimming.
DENIED_ROLES = frozenset(
    {
        "admin",
        "administrator",
        "owner",
        "system administrator",
        "global administrator",
        "power platform administrator",
    }
)

WILDCARD_SCOPE = "*"


def is_denied_role(role: Any) -> bool:
    if not role:
        return False
    return str(role).strip().lower() in DENIED_ROLES


def _has_recent_dry_run(
    binding_kind: str,
    scope_id: str,
    dry_runs: Iterable[dict],
    now: datetime,
) -> bool:
    """A successful dry run for this exact kind × scope, inside the window.

    Deliberately exact: a dry run against a lab workspace says nothing about
    production, and a dry run of a different binding kind says nothing at all.
    """
    cutoff = now - timedelta(days=DRY_RUN_VALIDITY_DAYS)
    for entry in dry_runs:
        if entry.get("binding_kind") != binding_kind:
            continue
        if entry.get("scope_id") != scope_id:
            continue
        succeeded_at = entry.get("succeeded_at")
        if not isinstance(succeeded_at, datetime):
            continue
        if succeeded_at.tzinfo is None:
            succeeded_at = succeeded_at.replace(tzinfo=timezone.utc)
        # A future timestamp is a clock problem, not an approval.
        if cutoff <= succeeded_at <= now:
            return True
    return False


def evaluate_write_gates(
    request: dict,
    config: dict,
    dry_runs: Iterable[dict] = (),
    now: datetime | None = None,
) -> dict:
    """Evaluate every gate. Returns `{"allowed": bool, "failed_gate": str|None, "detail": str|None}`.

    Order matters: the *first* refusal is what gets audited, so the unconditional
    invariants come first and the cheapest configuration checks follow. An audit
    row saying `gate:deniedRole` is a very different conversation from
    `gate:master`.
    """
    now = now or datetime.now(timezone.utc)
    dry_runs = list(dry_runs)

    binding_kind = request.get("binding_kind", "")
    module = request.get("module", "")
    scope_id = request.get("scope_id", "")
    role = request.get("role")
    is_dry_run = bool(request.get("dry_run", False))
    writable = bool(request.get("writable", False))

    def refuse(gate: str, detail: str | None = None) -> dict:
        return {"allowed": False, "failed_gate": gate, "detail": detail}

    # ── unconditional invariants — no configuration can override these ──────
    if is_denied_role(role):
        return refuse("deniedRole", f"role={role}")
    if module not in (config.get("enabled_modules") or []):
        return refuse("moduleOff", f"module={module}")
    if not writable:
        return refuse("notWritable", f"kind={binding_kind}")

    # ── the four gates ─────────────────────────────────────────────────────
    if not config.get("writes_enabled", False):
        return refuse("master", None)
    if binding_kind not in (config.get("armed_kinds") or []):
        return refuse("kind", f"kind={binding_kind}")

    # A dry run changes nothing, so it stops here. Requiring the scope
    # allow-list and a prior dry run of a dry run would make gate 4 unreachable
    # — you could never earn the dry run that unlocks the real write.
    if is_dry_run:
        return {"allowed": True, "failed_gate": None, "detail": None}

    allowlist = config.get("scope_allowlist") or []
    if WILDCARD_SCOPE not in allowlist and scope_id not in allowlist:
        return refuse("scope", f"scope={scope_id}")

    if not _has_recent_dry_run(binding_kind, scope_id, dry_runs, now):
        return refuse(
            "dryRun",
            f"kind={binding_kind} scope={scope_id} window={DRY_RUN_VALIDITY_DAYS}d",
        )

    return {"allowed": True, "failed_gate": None, "detail": None}

In [ ]:
# --- inlined from collectors/executors.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any, Callable

#: `(method, url, body) -> dict`. Injected so the executors never import a
#: transport and can be tested without one.
HttpCall = Callable[[str, str, dict | None], Any]

GRAPH = "https://graph.microsoft.com/v1.0"
FABRIC = "https://api.fabric.microsoft.com/v1"

#: Fabric workspace roles this tool will assign. `Admin` is absent on purpose
#: and is *also* blocked by the gate invariants — belt and braces, because this
#: is the one mistake that cannot be undone by the tool itself.
FABRIC_ROLES = ("Viewer", "Contributor", "Member")

#: How long before a verify pass should run. Group membership propagates fast;
#: workspace roles are read-your-writes but the app's collector is not.
VERIFY_AFTER_S = 900


def _need(binding: dict, *keys: str) -> None:
    missing = [k for k in keys if not binding.get(k)]
    if missing:
        raise ValueError(f"binding is missing {', '.join(missing)}")


def entra_group_member(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Add a principal to an Entra security group.

    The group is the one currency all four planes accept, so most entitlements
    reduce to exactly this call.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "target_id", "principal_id")
        group_id = binding["target_id"]
        principal_id = binding["principal_id"]

        # Read first: `POST /members/$ref` on an existing member returns 400
        # "One or more added object references already exist", which is
        # indistinguishable from a real failure at the call site.
        members = http("GET", f"{GRAPH}/groups/{group_id}/members?$select=id&$top=999", None)
        existing = {m.get("id") for m in (members or {}).get("value", []) or []}
        already = principal_id in existing

        before = {"is_member": already}
        if already:
            return {
                "ok": True,
                "before": before,
                "after": {"is_member": True},
                "detail": "already_present",
                "verified": True,  # we just read it — this one really is proven
            }

        if dry_run:
            return {
                "ok": True,
                "before": before,
                "after": {"is_member": True, "planned": True},
                "detail": f"would POST {GRAPH}/groups/{group_id}/members/$ref",
            }

        http(
            "POST",
            f"{GRAPH}/groups/{group_id}/members/$ref",
            {"@odata.id": f"{GRAPH}/directoryObjects/{principal_id}"},
        )
        return {
            "ok": True,
            "before": before,
            "after": {"is_member": True},
            "detail": "created",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def fabric_workspace_role(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Assign a workspace role to a principal (usually a group).

    Fabric has no per-item-type role, so `Contributor` here grants every create
    capability not separately gated by a tenant setting. That is a documented
    platform property, and the entitlement model already accounts for it — but
    it is why this executor refuses to invent a role it was not given.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "target_id", "principal_id", "role")
        workspace_id = binding["target_id"]
        principal_id = binding["principal_id"]
        role = binding["role"]
        principal_type = binding.get("principal_type") or "Group"

        if role not in FABRIC_ROLES:
            # Includes `Admin`. The gates refuse it too; this is the second lock.
            return {
                "ok": False,
                "error": f"role:{role} is not assignable by this tool",
                "detail": f"allowed roles: {', '.join(FABRIC_ROLES)}",
            }

        assignments = http(
            "GET", f"{FABRIC}/workspaces/{workspace_id}/roleAssignments", None
        )
        current = None
        for entry in (assignments or {}).get("value", []) or []:
            if ((entry.get("principal") or {}).get("id")) == principal_id:
                current = entry
                break

        before = {"role": current.get("role") if current else None}

        if current and current.get("role") == role:
            return {
                "ok": True,
                "before": before,
                "after": {"role": role},
                "detail": "already_present",
                "verified": True,
            }

        # A principal can hold only one role per workspace, so a change is a
        # PATCH of the existing assignment — POSTing again returns a conflict.
        if current:
            method, url, body = (
                "PATCH",
                f"{FABRIC}/workspaces/{workspace_id}/roleAssignments/{current.get('id')}",
                {"role": role},
            )
        else:
            method, url, body = (
                "POST",
                f"{FABRIC}/workspaces/{workspace_id}/roleAssignments",
                {"principal": {"id": principal_id, "type": principal_type}, "role": role},
            )

        if dry_run:
            return {
                "ok": True,
                "before": before,
                "after": {"role": role, "planned": True},
                "detail": f"would {method} {url}",
            }

        http(method, url, body)
        return {
            "ok": True,
            "before": before,
            "after": {"role": role},
            "detail": "changed" if current else "created",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def build_executors(graph_http: HttpCall | None, fabric_http: HttpCall | None) -> dict:
    """Register only the executors whose transport actually exists.

    A registered executor with no credential would fail at the HTTP call and be
    audited as `executor:failed` — which reads like the plane rejected us. Not
    registering it produces `executor:not-implemented`, which is the truth: this
    deployment cannot write there.
    """
    executors: dict[str, Callable[[dict, bool], dict]] = {}
    if graph_http is not None:
        executors["entra_group_member"] = entra_group_member(graph_http)
    if fabric_http is not None:
        executors["fabric_workspace_role"] = fabric_workspace_role(fabric_http)
    return executors

In [ ]:
# --- inlined from collectors/executors_pp.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any, Callable

#: `(method, url, body) -> dict`, injected — same contract as the other planes.
HttpCall = Callable[[str, str, dict | None], Any]

BAP = "https://api.bap.microsoft.com/providers/Microsoft.BusinessAppPlatform"
BAP_API_VERSION = "2021-04-01"

#: Environment types where a security group **cannot** be bound. Not a
#: permission problem — there is no supported way to do it (PLAN.md §8.6).
SG_IMPOSSIBLE_TYPES = ("Default", "Developer")

#: Dataverse roles this tool must never assign, whatever it is asked.
DENIED_DATAVERSE_ROLES = ("system administrator", "system customizer")

#: Connector classifications a data policy may use.
DLP_CLASSIFICATIONS = ("General", "Confidential", "Blocked")

VERIFY_AFTER_S = 900
#: Tenant settings propagate slowly enough that a synchronous check is
#: meaningless; the Fabric build learned this the hard way.
TENANT_VERIFY_AFTER_S = 3600


def _need(binding: dict, *keys: str) -> None:
    missing = [k for k in keys if not binding.get(k)]
    if missing:
        raise ValueError(f"binding is missing {', '.join(missing)}")


def pp_env_security_group(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Bind a Power Platform environment to an Entra security group.

    The single highest-leverage preventive control in Power Platform, and it
    costs nothing. Everything else in this module is containment around it.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "target_id", "principal_id")
        environment_id = binding["target_id"]
        group_id = binding["principal_id"]
        url = f"{BAP}/scopes/admin/environments/{environment_id}?api-version={BAP_API_VERSION}"

        current = http("GET", url, None) or {}
        properties = current.get("properties") or {}
        environment_type = (
            properties.get("environmentSku")
            or (properties.get("environmentType") or "")
        )
        existing = (
            (properties.get("linkedEnvironmentMetadata") or {}).get("securityGroupId")
            or properties.get("securityGroupId")
            or ""
        )

        if environment_type in SG_IMPOSSIBLE_TYPES:
            # Reported as a refusal with the real reason, not as an API error.
            # "You did not have permission" would send an admin hunting for a
            # role that would not have helped.
            return {
                "ok": False,
                "error": "platform:security-group-not-assignable",
                "detail": (
                    f"{environment_type} environments cannot be bound to a security group; "
                    "contain them with a data policy, tenant isolation and "
                    "disableShareWithEveryone instead"
                ),
                "before": {"security_group_id": existing},
            }

        if existing == group_id:
            return {
                "ok": True,
                "before": {"security_group_id": existing},
                "after": {"security_group_id": group_id},
                "detail": "already_present",
                "verified": True,
            }

        body = {"properties": {"linkedEnvironmentMetadata": {"securityGroupId": group_id}}}
        if dry_run:
            return {
                "ok": True,
                "before": {"security_group_id": existing},
                "after": {"security_group_id": group_id, "planned": True},
                "detail": f"would PATCH {url}",
            }

        http("PATCH", url, body)
        return {
            "ok": True,
            "before": {"security_group_id": existing},
            # Replacing an existing binding locks out the previous group's
            # members, so the audit row must show what was displaced.
            "after": {"security_group_id": group_id, "replaced": bool(existing)},
            "detail": "changed" if existing else "created",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def pp_dataverse_role(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Assign a Dataverse security role to an Entra **group team**.

    This is the binding that grants Copilot Studio agent authoring per
    environment — the one genuinely preventive agent control that exists, since
    agent creation cannot be disabled tenant-wide.

    It targets a *group team*, never a person. A role assigned to an individual
    is access no group membership explains: the Can-Do Explorer cannot derive
    it, and revoking it means hunting down every individual row.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "target_id", "principal_id", "role")
        env_url = (binding.get("environment_url") or "").rstrip("/")
        if not env_url:
            raise ValueError("binding is missing environment_url")

        role = binding["role"]
        if role.strip().lower() in DENIED_DATAVERSE_ROLES:
            return {
                "ok": False,
                "error": f"role:{role} is never assignable by this tool",
                "detail": "use a purpose-built role scoped to the tables the persona needs",
            }

        principal_type = (binding.get("principal_type") or "Team").strip()
        if principal_type.lower() not in ("team", "group"):
            return {
                "ok": False,
                "error": "principal:must-be-group-team",
                "detail": (
                    "Dataverse roles are only assigned to Entra group teams here. A role held "
                    "by an individual cannot be derived from any group membership, so it is "
                    "invisible to the entitlement model and hard to revoke."
                ),
            }

        api = f"{env_url}/api/data/v9.2"
        team_id = binding["principal_id"]
        role_id = binding.get("role_id")

        # Resolve the role by name when no id was compiled in. Roles are
        # per-environment, so a name is the portable identifier.
        if not role_id:
            escaped = role.replace("'", "''")
            found = http(
                "GET", f"{api}/roles?$select=roleid,name&$filter=name eq '{escaped}'", None
            )
            candidates = (found or {}).get("value", []) or []
            if not candidates:
                return {
                    "ok": False,
                    "error": "role:not-found",
                    "detail": f'no Dataverse role named "{role}" in this environment',
                }
            role_id = candidates[0].get("roleid")

        existing = http(
            "GET", f"{api}/teams({team_id})/teamroles_association?$select=roleid", None
        )
        held = {r.get("roleid") for r in (existing or {}).get("value", []) or []}
        if role_id in held:
            return {
                "ok": True,
                "before": {"has_role": True},
                "after": {"has_role": True},
                "detail": "already_present",
                "verified": True,
            }

        url = f"{api}/teams({team_id})/teamroles_association/$ref"
        body = {"@odata.id": f"{api}/roles({role_id})"}
        if dry_run:
            return {
                "ok": True,
                "before": {"has_role": False},
                "after": {"has_role": True, "planned": True, "role_id": role_id},
                "detail": f"would POST {url}",
            }

        http("POST", url, body)
        return {
            "ok": True,
            "before": {"has_role": False},
            "after": {"has_role": True, "role_id": role_id},
            "detail": "created",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def pp_tenant_setting(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Flip one governance-relevant tenant setting.

    Read-modify-write against the whole settings blob, because `Set-TenantSettings`
    replaces what it is given: a naive write of `{setting: value}` silently
    resets everything else in that section.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "setting_name")
        name = binding["setting_name"]
        value = binding.get("value")
        if value is None:
            raise ValueError("binding is missing value")
        path = binding.get("setting_path") or ["powerPlatform", "governance", name]

        current = http("POST", f"{BAP}/listtenantsettings?api-version={BAP_API_VERSION}", {}) or {}

        node: Any = current
        for key in path[:-1]:
            if not isinstance(node, dict):
                node = {}
                break
            node = node.get(key) or {}
        before = node.get(path[-1]) if isinstance(node, dict) else None

        if before == value:
            return {
                "ok": True,
                "before": {name: before},
                "after": {name: value},
                "detail": "already_present",
                "verified": True,
            }

        # Rebuild the full payload with one leaf changed.
        updated = _set_in(current, path, value)
        url = f"{BAP}/tenantsettings?api-version={BAP_API_VERSION}"
        if dry_run:
            return {
                "ok": True,
                "before": {name: before},
                "after": {name: value, "planned": True},
                "detail": f"would POST {url} ({'.'.join(path)})",
            }

        http("POST", url, updated)
        return {
            "ok": True,
            "before": {name: before},
            "after": {name: value},
            "detail": "changed",
            # Tenant settings take minutes; claiming otherwise is how a
            # governance tool reports a control that is not yet in force.
            "verified": False,
            "verify_after_s": TENANT_VERIFY_AFTER_S,
        }

    return execute


def _set_in(payload: dict, path: list[str], value: Any) -> dict:
    """Copy `payload` with `path` set to `value`, creating intermediate dicts."""
    import copy

    result = copy.deepcopy(payload) if payload else {}
    node = result
    for key in path[:-1]:
        nxt = node.get(key)
        if not isinstance(nxt, dict):
            nxt = {}
            node[key] = nxt
        node = nxt
    node[path[-1]] = value
    return result


def pp_tenant_isolation(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Turn cross-tenant isolation on (or off), with an allow-list.

    One of the six licence-free Default-environment levers. The API expresses
    the state as `isDisabled`, which is trivially easy to invert by accident —
    so the binding speaks in terms of `enabled` and the inversion happens here,
    once, where it is tested.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        enabled = binding.get("enabled")
        if enabled is None:
            raise ValueError("binding is missing enabled")
        allowed = binding.get("allowed_tenants") or []
        url = f"{BAP}/scopes/admin/tenantIsolationPolicy?api-version={BAP_API_VERSION}"

        current = http("GET", url, None) or {}
        properties = current.get("properties") or {}
        is_disabled = properties.get("isDisabled")
        before_enabled = None if is_disabled is None else (not bool(is_disabled))

        if before_enabled == bool(enabled) and not allowed:
            return {
                "ok": True,
                "before": {"enabled": before_enabled},
                "after": {"enabled": bool(enabled)},
                "detail": "already_present",
                "verified": True,
            }

        body = {
            "properties": {
                "isDisabled": not bool(enabled),
                "allowedTenants": allowed,
            }
        }
        if dry_run:
            return {
                "ok": True,
                "before": {"enabled": before_enabled},
                "after": {"enabled": bool(enabled), "planned": True},
                "detail": f"would PUT {url}",
            }

        http("PUT", url, body)
        return {
            "ok": True,
            "before": {"enabled": before_enabled},
            "after": {"enabled": bool(enabled), "allowed_tenants": len(allowed)},
            "detail": "changed",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def pp_data_policy(http: HttpCall) -> Callable[[dict, bool], dict]:
    """Create or update a data policy — the primary licence-free lever.

    Data policies have **no licence prerequisite** and do not need Managed
    Environments, which is why this is the control the Default-environment
    posture score leans on.
    """

    def execute(binding: dict, dry_run: bool) -> dict:
        _need(binding, "policy_name")
        name = binding["policy_name"]
        default_group = binding.get("default_connector_group") or "Blocked"
        if default_group not in DLP_CLASSIFICATIONS:
            return {
                "ok": False,
                "error": f"classification:{default_group} is not a valid connector group",
                "detail": f"expected one of {', '.join(DLP_CLASSIFICATIONS)}",
            }

        environments = binding.get("environments") or []
        base = f"{BAP}/scopes/admin/v2/policies"
        existing_payload = http("GET", f"{base}?api-version={BAP_API_VERSION}", None) or {}
        existing = None
        for policy in existing_payload.get("value", []) or []:
            if policy.get("displayName") == name:
                existing = policy
                break

        before = (
            {
                "policy_id": existing.get("name"),
                "default_connector_group": existing.get("defaultConnectorClassification"),
            }
            if existing
            else {"policy_id": None}
        )

        if existing and existing.get("defaultConnectorClassification") == default_group:
            return {
                "ok": True,
                "before": before,
                "after": {"default_connector_group": default_group},
                "detail": "already_present",
                "verified": True,
            }

        body = {
            "displayName": name,
            "defaultConnectorClassification": default_group,
            "connectorGroups": binding.get("connector_groups") or [],
            "environmentType": "OnlyEnvironments" if environments else "AllEnvironments",
            "environments": [{"name": e} for e in environments],
        }

        if existing:
            method = "PATCH"
            url = f"{base}/{existing.get('name')}?api-version={BAP_API_VERSION}"
        else:
            method = "POST"
            url = f"{base}?api-version={BAP_API_VERSION}"

        if dry_run:
            return {
                "ok": True,
                "before": before,
                "after": {"default_connector_group": default_group, "planned": True},
                "detail": f"would {method} {url}",
            }

        created = http(method, url, body) or {}
        return {
            "ok": True,
            "before": before,
            "after": {
                "policy_id": created.get("name") or (existing or {}).get("name"),
                "default_connector_group": default_group,
            },
            "detail": "changed" if existing else "created",
            "verified": False,
            "verify_after_s": VERIFY_AFTER_S,
        }

    return execute


def build_pp_executors(
    bap_http: HttpCall | None,
    dataverse_http: HttpCall | None = None,
) -> dict:
    """Register the PP executors whose transport exists.

    `pp_dataverse_role` needs a Dataverse token per environment, which is a
    different credential from the BAP admin one — so it is registered
    separately rather than being assumed to work because BAP does.
    """
    executors: dict[str, Callable[[dict, bool], dict]] = {}
    if bap_http is not None:
        executors["pp_env_security_group"] = pp_env_security_group(bap_http)
        executors["pp_tenant_setting"] = pp_tenant_setting(bap_http)
        executors["pp_tenant_isolation"] = pp_tenant_isolation(bap_http)
        executors["pp_data_policy"] = pp_data_policy(bap_http)
    if dataverse_http is not None:
        executors["pp_dataverse_role"] = pp_dataverse_role(dataverse_http)
    return executors

In [ ]:
# --- inlined from collectors/actuator.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable


#: Executors are registered per binding kind by the plane modules in later
#: phases. A kind with no executor is refused, loudly — never silently treated
#: as a success.
Executor = Callable[[dict, bool], dict]


def _utcnow() -> datetime:
    return datetime.now(timezone.utc)


def _as_json(value: Any) -> str | None:
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def build_audit_row(
    *,
    actor: str,
    actor_type: str,
    action: str,
    plane: str,
    target_type: str,
    target_id: str,
    outcome: str,
    request_id: str = "",
    correlation_id: str = "",
    before: Any = None,
    after: Any = None,
    error: str | None = None,
    ts: datetime | None = None,
) -> dict:
    """One `gov_audit` row. Append-only; this table is the real deliverable."""
    return {
        "audit_id": str(uuid.uuid4()),
        "ts": ts or _utcnow(),
        "actor": actor,
        "actor_type": actor_type,
        "action": action,
        "plane": plane,
        "target_type": target_type,
        "target_id": target_id,
        "before_json": _as_json(before),
        "after_json": _as_json(after),
        "request_id": request_id,
        "correlation_id": correlation_id,
        "outcome": outcome,
        "error": error,
    }


def result(
    *,
    ok: bool,
    dry_run: bool,
    before: Any = None,
    after: Any = None,
    verified: bool = False,
    verify_after_s: int = 0,
    detail: str = "",
    error: str | None = None,
) -> dict:
    """The actuator `exitValue` contract (PLAN.md §14)."""
    return {
        "ok": ok,
        "dry_run": dry_run,
        "before": before,
        "after": after,
        "verified": verified,
        "verify_after_s": verify_after_s,
        "detail": detail,
        "error": error,
    }


def run_actuator(
    request: dict,
    config: dict,
    dry_runs: list[dict],
    executors: dict[str, Executor],
    *,
    now: datetime | None = None,
) -> dict:
    """Decide, execute, and produce everything the notebook must persist.

    Returns `{"result": …, "audit": row, "dry_run_row": row|None}`. The caller
    writes `audit` unconditionally — that is the point of returning it rather
    than writing it here.
    """
    now = now or _utcnow()
    binding = request.get("binding") or {}
    binding_kind = binding.get("kind", "")
    scope_id = binding.get("target_id", "")
    is_dry_run = bool(request.get("dry_run", True))
    actor = request.get("actor") or "unknown"
    action = f"{'dryrun' if is_dry_run else 'write'}:{binding_kind}"
    plane = binding.get("module", "")

    audit = lambda outcome, error=None, before=None, after=None: build_audit_row(  # noqa: E731
        actor=actor,
        actor_type=request.get("actor_type", "User"),
        action=action,
        plane=plane,
        target_type=binding.get("target_type", "Scope"),
        target_id=scope_id,
        outcome=outcome,
        request_id=request.get("request_id", ""),
        correlation_id=request.get("correlation_id", ""),
        before=before,
        after=after,
        error=error,
        ts=now,
    )

    decision = evaluate_write_gates(
        {
            "binding_kind": binding_kind,
            "module": plane,
            "scope_id": scope_id,
            "role": binding.get("role"),
            "dry_run": is_dry_run,
            "writable": bool(binding.get("writable", False)),
        },
        config,
        dry_runs,
        now=now,
    )

    if not decision["allowed"]:
        gate = decision["failed_gate"]
        return {
            "result": result(
                ok=False,
                dry_run=is_dry_run,
                detail=decision.get("detail") or "",
                error=f"gate:{gate}",
            ),
            "audit": audit("Refused", error=f"gate:{gate} {decision.get('detail') or ''}".strip()),
            "dry_run_row": None,
        }

    executor = executors.get(binding_kind)
    if executor is None:
        # Honest failure. A binding kind with no executor is a plane this build
        # cannot write to yet — returning ok would claim a grant that does not
        # exist, and the drift engine would then report the *platform* as wrong.
        return {
            "result": result(
                ok=False,
                dry_run=is_dry_run,
                detail=f"no executor registered for {binding_kind}",
                error="executor:not-implemented",
            ),
            "audit": audit("Failed", error=f"executor:not-implemented kind={binding_kind}"),
            "dry_run_row": None,
        }

    try:
        outcome = executor(binding, is_dry_run) or {}
    except Exception as exc:  # noqa: BLE001 — the audit row is the whole point
        return {
            "result": result(
                ok=False,
                dry_run=is_dry_run,
                detail=f"{type(exc).__name__}: {exc}",
                error="executor:failed",
            ),
            "audit": audit("Failed", error=f"{type(exc).__name__}: {exc}"),
            "dry_run_row": None,
        }

    ok = bool(outcome.get("ok", True))
    before = outcome.get("before")
    after = outcome.get("after")
    detail = outcome.get("detail", "")

    # Only a *successful* dry run earns gate-4 credit.
    dry_run_row = None
    if ok and is_dry_run:
        dry_run_row = {
            "binding_kind": binding_kind,
            "scope_id": scope_id,
            "succeeded_at": now,
            "actor": actor,
            "correlation_id": request.get("correlation_id", ""),
        }

    return {
        "result": result(
            ok=ok,
            dry_run=is_dry_run,
            before=before,
            after=after,
            verified=bool(outcome.get("verified", False)),
            verify_after_s=int(outcome.get("verify_after_s", 0)),
            detail=detail,
            error=outcome.get("error"),
        ),
        "audit": audit(
            "Planned" if is_dry_run and ok else ("Success" if ok else "Failed"),
            error=outcome.get("error"),
            before=before,
            after=after,
        ),
        "dry_run_row": dry_run_row,
    }

In [ ]:
import json

try:
    import notebookutils  # type: ignore
except ImportError:  # pragma: no cover - local syntax checks
    notebookutils = None

try:
    spark  # type: ignore[name-defined]
except NameError:  # pragma: no cover - local syntax checks
    spark = None

steps = []


def log(step, status, detail=""):
    steps.append({"step": step, "status": status, "detail": detail})
    print(f"[{status:>22}] {step}{(' — ' + detail) if detail else ''}")


request = json.loads(request_json or "{}")
binding = request.get("binding") or {}
log(
    "request",
    "Parsed",
    f"kind={binding.get('kind')} scope={binding.get('target_id')} "
    f"dry_run={request.get('dry_run', True)}",
)

## Read the gate configuration — from the lakehouse, never from the caller

The request carries *what* to do. It never carries permission to do it: the
configuration is read here, server-side, on every call. A caller that could
supply its own `writes_enabled` would have no gates at all.

In [ ]:
def _config_value(key, default):
    if spark is None:
        return default
    try:
        rows = spark.sql(
            f"SELECT config_value FROM {lakehouse_name}.gov_config "  # noqa: S608 - fixed table
            f"WHERE config_key = '{key}'"
        ).collect()
    except Exception as exc:  # noqa: BLE001
        log(f"config:{key}", "Failed", f"{type(exc).__name__}: {exc}")
        return default
    if not rows:
        return default
    try:
        return json.loads(rows[0]["config_value"])
    except (TypeError, ValueError):
        return rows[0]["config_value"]


config = {
    "writes_enabled": bool(_config_value("writes.enabled", False)),
    "armed_kinds": _config_value("writes.kinds", []) or [],
    "scope_allowlist": _config_value("writes.scopeAllowlist", []) or [],
    "enabled_modules": _config_value("modules.enabled", []) or [],
}
log(
    "config",
    "Read",
    f"writes_enabled={config['writes_enabled']} "
    f"kinds={len(config['armed_kinds'])} scopes={len(config['scope_allowlist'])}",
)

## Gate 4 evidence — prior successful dry runs

Read narrowly: this binding kind, this scope, inside the 30-day window. Gate 4
is what turns *"we tested it"* from a claim into a machine fact.

In [ ]:
dry_runs = []
if spark is not None:
    try:
        rows = spark.sql(
            f"SELECT binding_kind, scope_id, succeeded_at "  # noqa: S608 - fixed table
            f"FROM {lakehouse_name}.gov_dry_runs"
        ).collect()
        dry_runs = [
            {
                "binding_kind": r["binding_kind"],
                "scope_id": r["scope_id"],
                "succeeded_at": r["succeeded_at"],
            }
            for r in rows
        ]
    except Exception as exc:  # noqa: BLE001
        # An unreadable ledger must fail *closed*: no evidence means no write.
        log("dry_runs", "Failed", f"{type(exc).__name__}: {exc}")
log("dry_runs", "Read", f"{len(dry_runs)} recorded")

## Executors

Only the executors whose transport actually exists are registered. A
registered executor with no credential would fail at the HTTP call and be
audited as `executor:failed` — which reads like the plane rejected us. Leaving
it unregistered produces `executor:not-implemented`, which is the truth: this
deployment cannot write there.

The Power Platform set is the **licence-free** one (PLAN.md §8.5): environment
security group, Dataverse role via a group team, data policy, tenant isolation
and the environment-creation tenant settings. `pp_managed_env` is deliberately
**not** registered — enabling Managed Environments makes premium licences a
requirement for active usage, and a governance tool must never trigger that as
a side effect of granting somebody access.

In [ ]:
graph_http = None
fabric_http = None
bap_http = None
dataverse_http = None

try:
    _graph_token = graph_token()
    graph_http = lambda method, url, body: graph_call(_graph_token, method, url, body)  # noqa: E731
    log("transport:graph", "Ready", "delegated token acquired")
except Exception as exc:  # noqa: BLE001
    log("transport:graph", "Unavailable", f"{type(exc).__name__}: {exc}")

try:
    _fabric_client = fabric_client()
    fabric_http = lambda method, url, body: fabric_call(  # noqa: E731
        _fabric_client, method, url.replace(FABRIC, ""), body
    )
    log("transport:fabric", "Ready", "sempy client")
except Exception as exc:  # noqa: BLE001
    log("transport:fabric", "Unavailable", f"{type(exc).__name__}: {exc}")


def _sp_token(resource):
    """Client-credentials token for the Power Platform management app.

    The FULL Key Vault URI is required — a short name fails with
    'Invalid vault uri' (learned in the Data Catalog build).
    """
    import urllib.parse
    import urllib.request

    client_id = notebookutils.credentials.getSecret(key_vault_uri, sp_client_id_secret)
    client_secret = notebookutils.credentials.getSecret(key_vault_uri, sp_secret_secret)
    data = urllib.parse.urlencode(
        {
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": f"{resource}/.default",
            "grant_type": "client_credentials",
        }
    ).encode()
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    with urllib.request.urlopen(  # noqa: S310 - fixed host
        urllib.request.Request(url, data=data)
    ) as response:
        return json.loads(response.read().decode("utf-8"))["access_token"]


if key_vault_uri and tenant_id and notebookutils is not None:
    try:
        _bap_token = _sp_token("https://service.powerapps.com")
        bap_http = lambda method, url, body: graph_call(_bap_token, method, url, body)  # noqa: E731
        log("transport:bap", "Ready", "management app token")
    except Exception as exc:  # noqa: BLE001
        log("transport:bap", "Unavailable", f"{type(exc).__name__}: {exc}")

    # Dataverse is a *different* audience, per environment — so it is acquired
    # separately rather than assumed to work because BAP does.
    _env_url = (binding.get("environment_url") or "").rstrip("/")
    if _env_url:
        try:
            _dv_token = _sp_token(_env_url)
            dataverse_http = lambda method, url, body: graph_call(  # noqa: E731
                _dv_token, method, url, body
            )
            log("transport:dataverse", "Ready", _env_url)
        except Exception as exc:  # noqa: BLE001
            log("transport:dataverse", "Unavailable", f"{type(exc).__name__}: {exc}")
else:
    log("transport:pp", "Unavailable", "key_vault_uri / tenant_id not configured")

EXECUTORS = build_executors(graph_http, fabric_http)
EXECUTORS.update(build_pp_executors(bap_http, dataverse_http))

log("executors", "Registered", ", ".join(sorted(EXECUTORS)) or "none (framework only)")

## Decide, execute, audit

`run_actuator` is pure and unit-tested offline. This cell is only IO.

In [ ]:
outcome = run_actuator(request, config, dry_runs, EXECUTORS)
audit_row = outcome["audit"]
log("decision", audit_row["outcome"], outcome["result"].get("error") or "")

if persist_audit and spark is not None:
    try:
        spark.createDataFrame([audit_row]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse_name}.gov_audit")
        log("gov_audit", "Appended", audit_row["audit_id"])
    except Exception as exc:  # noqa: BLE001
        log("gov_audit", "Failed", f"{type(exc).__name__}: {exc}")

if outcome["dry_run_row"] is not None and spark is not None:
    try:
        spark.createDataFrame([outcome["dry_run_row"]]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse_name}.gov_dry_runs")
        log("gov_dry_runs", "Appended", "gate 4 credit granted")
    except Exception as exc:  # noqa: BLE001
        log("gov_dry_runs", "Failed", f"{type(exc).__name__}: {exc}")

In [ ]:
exit_value = json.dumps(outcome["result"], default=str)
print(exit_value)
if notebookutils is not None:
    notebookutils.notebook.exit(exit_value)